# Feature Engineering

## Principal Component Analysis - PCA

#### Why do we need PCA?
- Dimensionality Reduction (Example: 4D to 2D, etc.)
- Anomaly Detection
- Noise Reduction
- Decorrelation

In [19]:
from sklearn.decomposition import PCA
from IPython.display import display
from sklearn.feature_selection import mutual_info_regression

import pandas as pd

def make_mi_scores(X, y, discrete_features):
    mi_scores = mutual_info_regression(X, y, discrete_features=discrete_features)
    mi_scores = pd.Series(mi_scores, name="MI Scores", index=X.columns)
    mi_scores = mi_scores.sort_values(ascending=False)
    return mi_scores

In [2]:
autos_data = pd.read_csv('../../resources/autos.csv')
autos_data.head()

,symboling,make,fuel_type,aspiration,num_of_doors,body_style,drive_wheels,engine_location,wheel_base,length,...,engine_size,fuel_system,bore,stroke,compression_ratio,horsepower,peak_rpm,city_mpg,highway_mpg,price
0,3,alfa-romero,gas,std,2,convertible,rwd,front,88.6,168.8,...,130,mpfi,3.47,2.68,9,111,5000,21,27,13495
1,3,alfa-romero,gas,std,2,convertible,rwd,front,88.6,168.8,...,130,mpfi,3.47,2.68,9,111,5000,21,27,16500
2,1,alfa-romero,gas,std,2,hatchback,rwd,front,94.5,171.2,...,152,mpfi,2.68,3.47,9,154,5000,19,26,16500
3,2,audi,gas,std,4,sedan,fwd,front,99.8,176.6,...,109,mpfi,3.19,3.40,10,102,5500,24,30,13950
4,2,audi,gas,std,4,sedan,4wd,front,99.4,176.6,...,136,mpfi,3.19,3.40,8,115,5500,18,22,17450


### Step 1 - Standardize Data

Before implementing PCA, standardize data for best results.

In [11]:
X = autos_data.copy()
y = X.pop('price')

# Standardize continuous numerical features
X_continuous = X.select_dtypes(include=['int', 'float'])

# Factorize categorical features
X_categorical = X.select_dtypes(include=['object', "str"])
for cat_col in X_categorical:
    X_categorical[cat_col], _ = X_categorical[cat_col].factorize()

# Create a new dataframe with standardized features
X_new = X_continuous.join(X_categorical)

# Scale features
X_scaled = (X_new - X_new.mean(axis=0)) / X_new.std(axis=0)
display(X_scaled.head())

,make,fuel_type,aspiration,body_style,drive_wheels,engine_location,engine_type,fuel_system
0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0
2,0,0,0,1,0,0,1,0
3,1,0,0,2,1,0,2,0
4,1,0,0,2,2,0,2,0


,symboling,num_of_doors,wheel_base,length,width,height,curb_weight,num_of_cylinders,engine_size,bore,...,city_mpg,highway_mpg,make,fuel_type,aspiration,body_style,drive_wheels,engine_location,engine_type,fuel_system
0,1.782215,-1.172839,-1.678015,-0.442872,-0.839080,-2.117092,-0.025646,-0.410180,0.045098,0.511697,...,-0.677292,-0.555613,-2.038639,-0.32959,-0.469437,-2.155714,-1.220954,-0.12533,-2.435319,-0.708026
1,1.782215,-1.172839,-1.678015,-0.442872,-0.839080,-2.117092,-0.025646,-0.410180,0.045098,0.511697,...,-0.677292,-0.555613,-2.038639,-0.32959,-0.469437,-2.155714,-1.220954,-0.12533,-2.435319,-0.708026
2,0.163544,-1.172839,-0.719041,-0.250543,-0.184200,-0.613816,0.496473,1.544506,0.574066,-2.388614,...,-0.990387,-0.702307,-2.038639,-0.32959,-0.469437,-0.970378,-1.220954,-0.12533,-1.233231,-0.708026
3,0.972880,0.848214,0.142410,0.182198,0.143240,0.179580,-0.426254,-0.410180,-0.459826,-0.516262,...,-0.207649,-0.115531,-1.869264,-0.32959,-0.469437,0.214957,0.591693,-0.12533,-0.031142,-0.708026
4,0.972880,0.848214,0.077395,0.182198,0.236794,0.179580,0.498371,0.567163,0.189362,-0.516262,...,-1.146935,-1.289083,-1.869264,-0.32959,-0.469437,0.214957,2.404340,-0.12533,-0.031142,-0.708026


### Step 2 - Apply PCA

Apply the Principal component analysis on standardized features.

In [16]:
pca = PCA()
X_new_pca = pca.fit_transform(X_new, y)
print(X_new_pca.shape)

(193, 24)


In [17]:
# Convert to DataFrame
components = [f"PC{i+1}" for i in range(X_new_pca.shape[1])]
X_new_pca_df = pd.DataFrame(X_new_pca, columns=components)
display(X_new_pca_df.head())

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC15,PC16,PC17,PC18,PC19,PC20,PC21,PC22,PC23,PC24
0,43.593207,-90.228217,11.999919,-5.648686,-10.742782,-11.620323,-3.034289,5.361047,-5.909475,-0.505375,...,-0.886794,-0.212624,0.010893,-0.886531,-0.439188,-0.726481,-0.205961,-0.075407,0.089472,-0.010407
1,43.593207,-90.228217,11.999919,-5.648686,-10.742782,-11.620323,-3.034289,5.361047,-5.909475,-0.505375,...,-0.886794,-0.212624,0.010893,-0.886531,-0.439188,-0.726481,-0.205961,-0.075407,0.089472,-0.010407
2,275.780024,62.706163,32.016061,-20.560136,-11.670552,-10.269542,3.158006,0.424472,-0.551852,1.943163,...,-1.077360,-0.092255,-0.890716,0.659535,-0.164202,0.566808,-0.031073,-0.284768,-0.185224,-0.014316
3,-407.475291,211.826601,-2.715515,-2.961968,4.071367,-11.171339,6.270028,2.065904,0.874693,0.568370,...,0.324586,0.574938,1.094958,-0.216128,0.068164,0.150219,-0.181368,0.036975,-0.039780,-0.090483
4,0.605822,478.608820,-17.786199,3.690758,-5.589410,-10.822610,-3.835079,2.163376,0.485726,1.567163,...,0.268844,0.748655,1.431451,0.345134,0.881279,0.168140,-0.243431,0.073022,0.029666,0.036093


In [18]:
loadings = pd.DataFrame(
    pca.components_.T,
    columns=components,
    index=X_new.columns
)
display(loadings)

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC15,PC16,PC17,PC18,PC19,PC20,PC21,PC22,PC23,PC24
symboling,-0.000609,0.000163,0.014955,-0.010930,-0.056448,0.002027,-0.035939,0.138510,-0.144900,-0.107959,...,0.251094,0.302477,0.490275,0.080368,-0.133478,0.012465,-0.010336,0.029628,-0.004542,0.003770
num_of_doors,0.000438,-0.000105,-0.014696,0.001850,0.053532,-0.006337,0.023768,-0.041991,0.012242,0.122079,...,-0.164889,0.205855,0.703450,-0.132813,-0.242955,-0.001960,-0.003350,-0.002568,-0.013134,0.000877
wheel_base,0.008255,0.003114,-0.065098,0.067011,0.343892,-0.084566,0.196724,-0.392251,0.718585,-0.278204,...,0.086707,0.050742,0.085212,0.066718,0.039069,-0.005251,-0.027239,0.005401,0.007800,-0.001588
length,0.017725,0.010508,-0.071967,-0.010790,0.803406,-0.155743,0.341257,0.213327,-0.380389,-0.039976,...,0.018912,-0.052631,-0.007701,0.017478,0.003479,0.005719,0.013270,-0.001669,0.000117,0.000641
width,0.002946,0.001886,-0.003886,0.012195,0.060585,-0.021034,0.054167,0.003644,0.079879,-0.196132,...,-0.324053,0.127694,-0.205970,-0.144341,-0.100286,-0.011658,-0.044031,-0.011379,0.006496,-0.002023
height,0.001478,-0.000182,-0.044672,0.000857,0.137227,0.002498,0.052066,-0.131272,0.220412,0.842530,...,-0.158082,-0.050732,-0.019266,-0.036862,-0.070757,-0.000842,0.012891,0.007411,-0.008471,-0.000283
curb_weight,0.833688,0.543852,-0.089693,-0.003351,-0.030803,0.006200,-0.006469,-0.007434,-0.006741,0.000390,...,0.000736,0.000118,-0.000119,0.000671,0.000484,0.000210,-0.000062,0.000080,0.000093,-0.000003
num_of_cylinders,0.000934,0.000893,0.019866,0.022065,-0.005500,0.000743,0.001113,-0.000252,-0.001256,0.055092,...,-0.175959,0.097876,-0.024974,0.867865,-0.175680,0.164352,0.170005,0.269374,0.075391,0.025366
engine_size,0.055851,0.038968,0.699951,0.694652,0.105136,0.062271,-0.071237,0.031302,0.002421,0.019796,...,0.002742,-0.011132,0.005251,-0.023986,0.003751,-0.010099,0.006880,-0.008535,-0.003059,-0.000568
bore,0.000299,0.000124,0.001394,-0.003912,0.008001,0.004336,-0.008140,0.000201,0.001752,-0.001879,...,0.108569,-0.099713,-0.004707,-0.205582,-0.027733,-0.255236,-0.087347,0.888087,0.268653,0.016968


### Step 3 - Lookup Mutual Info scores between features

In [20]:
mi_scores = make_mi_scores(X_new_pca_df, y, discrete_features=False)
display(mi_scores)

PC1     0.576434
PC4     0.521397
PC2     0.351795
PC3     0.307288
PC22    0.175446
PC23    0.170820
PC20    0.169214
PC11    0.160474
PC7     0.158447
PC24    0.143731
PC21    0.138321
PC18    0.133082
PC5     0.131171
PC6     0.130125
PC10    0.125720
PC14    0.114958
PC9     0.087892
PC19    0.083882
PC16    0.069956
PC13    0.043505
PC8     0.024647
PC12    0.021416
PC17    0.013890
PC15    0.000000
Name: MI Scores, dtype: float64